In [1]:
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)

0.3.25
0.3.24


In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate

import utils

In [3]:
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

In [4]:
@tool
def add(x: float, y: float) -> float:
    """Add 'x' and 'y'."""
    return x + y

@tool
def multiply(x: float, y: float) -> float:
    """Multiply 'x' and 'y'."""
    return x * y

@tool
def exponentiate(x: float, y: float) -> float:
    """Raise 'x' to the power of 'y'."""
    return x ** y

@tool
def subtract(x: float, y: float) -> float:
    """Subtract 'x' from 'y'."""
    return y - x

In [5]:
new_prompt = ChatPromptTemplate.from_messages([
    ("system", "You have access to distinct math tools. When a user asks for a sequence of operations, you must perform them one by one, feeding the output of the previous tool into the next tool. Do not provide a final answer until every operation requested by the user has been executed."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

In [6]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
agent = create_tool_calling_agent(llm=model, tools=[add, multiply, exponentiate, subtract], prompt=new_prompt)
executor = AgentExecutor(agent=agent, tools=[add, multiply, exponentiate, subtract], verbose=True)

In [7]:
new_output = executor.invoke({"input": "What is the result of adding 5 and 3, multiplying by 2, then raising the result to the power of 2?"})
print("New Agent Output:", new_output)



> Entering new AgentExecutor chain...

Invoking: `add` with `{'x': 5, 'y': 3}`


8.0
Invoking: `multiply` with `{'x': 8, 'y': 2}`


16.0
Invoking: `exponentiate` with `{'x': 16, 'y': 2}`


256.0The final result of the operations is 256.

> Finished chain.
New Agent Output: {'input': 'What is the result of adding 5 and 3, multiplying by 2, then raising the result to the power of 2?', 'output': 'The final result of the operations is 256.'}
